# Czech bank data — PRAGMA-lite exploratory analysis and split

This notebook uses the row-level tables in `financial_db_Teradata/` directly. Teradata and BTEQ are not required: the source `.tsv` files are enough for PyTorch.

For version one, an **account** is one modelled history. This preserves the natural transaction ledger without duplicating a jointly held account across multiple clients.

## Questions

1. How long and how dense is each account history?
2. Which transaction fields are categorical, numeric, or identifiers to exclude?
3. Can we attach only static, cutoff-safe account/client/district attributes as profile state?
4. Does a fixed account-disjoint train/validation/test split prevent account leakage?

The later tokenizer must fit amount percentiles and categorical vocabularies from the training split only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda value: f'{value:,.3f}')

ROOT = Path.cwd()
DATA_DIR = ROOT / 'financial_db_Teradata'
OUTPUT_DIR = ROOT / 'data' / 'processed' / 'czech_bank'
RANDOM_SEED = 42

assert DATA_DIR.exists(), f'Cannot find {DATA_DIR}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load the source tables

The repository stores headerless, tab-separated files. Column names below come from the published Teradata schema.

In [ ]:
TRANS_COLUMNS = [
    'trans_id', 'account_id', 'trans_date', 'amount', 'balance',
    'trans_type', 'operation', 'category', 'other_bank_id', 'other_account_id',
]
ACCOUNT_COLUMNS = ['account_id', 'district_id', 'create_date', 'frequency']
DISP_COLUMNS = ['disp_id', 'client_id', 'account_id', 'disp_type']
CLIENT_COLUMNS = ['client_id', 'birth_date', 'gender', 'district_id']
DISTRICT_COLUMNS = [
    'district_id', 'district_name', 'region', 'num_inhabitants',
    'num_municipalities_gt499', 'num_municipalities_500to1999',
    'num_municipalities_2000to9999', 'num_municipalities_gt10000',
    'num_cities', 'ratio_urban', 'average_salary', 'unemployment_rate95',
    'unemployment_rate96', 'num_entrep_per1000', 'num_crimes95', 'num_crimes96',
]

def read_tsv(name, columns, parse_dates=()):
    path = DATA_DIR / name
    assert path.exists(), f'Missing {path}'
    return pd.read_csv(
        path, sep='\t', header=None, names=columns,
        na_values=[''], keep_default_na=True, parse_dates=list(parse_dates), low_memory=False,
    )

transactions = read_tsv('fin_trans.tsv', TRANS_COLUMNS, parse_dates=['trans_date'])
accounts = read_tsv('fin_account.tsv', ACCOUNT_COLUMNS, parse_dates=['create_date'])
dispositions = read_tsv('fin_disp.tsv', DISP_COLUMNS)
clients = read_tsv('fin_client.tsv', CLIENT_COLUMNS, parse_dates=['birth_date'])
districts = read_tsv('fin_district.tsv', DISTRICT_COLUMNS)

print(f'Transactions: {len(transactions):,}')
print(f'Accounts: {len(accounts):,}')
print(f'Clients: {len(clients):,}')
display(transactions.head())

## Canonical transaction events

`trans_id` and `other_account_id` are identifiers and are never model inputs. Blank operation/category/bank fields are kept as missing values; the tokenizer will later map them to a dedicated `[MISSING]` value token rather than pretending they are a valid category.

In [ ]:
ACCOUNT_ID = 'account_id'
TIMESTAMP = 'trans_date'
NUMERIC_FIELDS = ['amount', 'balance']
CATEGORICAL_FIELDS = ['trans_type', 'operation', 'category', 'other_bank_id']
DROP_FROM_MODEL = ['trans_id', 'other_account_id']

for field in CATEGORICAL_FIELDS:
    transactions[field] = transactions[field].astype('string').str.strip().replace('', pd.NA)

transactions = (
    transactions.dropna(subset=[ACCOUNT_ID, TIMESTAMP, *NUMERIC_FIELDS])
                .drop_duplicates(subset=['trans_id'])
                .sort_values([ACCOUNT_ID, TIMESTAMP, 'trans_id'])
                .reset_index(drop=True)
)

quality = pd.DataFrame({
    'missing_rows': transactions[[ACCOUNT_ID, TIMESTAMP, *NUMERIC_FIELDS, *CATEGORICAL_FIELDS]].isna().sum(),
    'missing_%': 100 * transactions[[ACCOUNT_ID, TIMESTAMP, *NUMERIC_FIELDS, *CATEGORICAL_FIELDS]].isna().mean(),
})
display(quality)
print(f'Duplicate transaction IDs removed: {len(transactions) - transactions.trans_id.nunique():,}')
display(transactions[NUMERIC_FIELDS].describe(percentiles=[.01, .05, .5, .95, .99]).T)

for field in CATEGORICAL_FIELDS:
    print(f'\n{field} ({transactions[field].nunique(dropna=True)} non-missing values)')
    display(transactions[field].value_counts(dropna=False).rename('events').to_frame())

## Account histories

The first chart uses logarithmically spaced bins, so the visual widths are honest on the log-scaled x-axis.

In [ ]:
history = transactions.groupby(ACCOUNT_ID).agg(
    n_events=('trans_id', 'size'),
    first_event=(TIMESTAMP, 'min'),
    last_event=(TIMESTAMP, 'max'),
    total_amount=('amount', 'sum'),
).reset_index()
history['history_span_days'] = (history['last_event'] - history['first_event']).dt.total_seconds() / 86_400

display(history[['n_events', 'history_span_days', 'total_amount']].describe(
    percentiles=[.01, .05, .1, .25, .5, .75, .9, .95, .99]
))

event_bins = np.geomspace(1, history['n_events'].max() + 1, 50)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(history['n_events'], bins=event_bins)
axes[0].set_xscale('log')
axes[0].set_title('Events per account')
axes[0].set_xlabel('Number of events (log scale)')
axes[0].set_ylabel('Accounts')
axes[1].hist(history['history_span_days'], bins=50)
axes[1].set_title('Elapsed account-history span')
axes[1].set_xlabel('Days between first and last transaction')
axes[1].set_ylabel('Accounts')
plt.tight_layout()

## Profile state, joined safely

We use the account owner (`disp_type == 'O'`) to avoid duplicating one shared account's ledger for every disponent. Account frequency, owner demographics, and district data are static. Loan/card information is intentionally excluded for now because it must be included only when known before a sample's prediction cutoff.

In [ ]:
owners = dispositions.loc[dispositions['disp_type'].eq('O'), ['account_id', 'client_id']].copy()
assert owners['account_id'].is_unique, 'Expected exactly one owner per account; investigate before continuing.'

client_profile = clients.rename(columns={'district_id': 'client_district_id'})
profile = (
    accounts.merge(owners, on='account_id', how='left', validate='one_to_one')
            .merge(client_profile, on='client_id', how='left', validate='many_to_one')
            .merge(districts, left_on='client_district_id', right_on='district_id', how='left',
                   suffixes=('', '_district'), validate='many_to_one')
)

# Keep date of birth, not a globally calculated age. The dataset code will calculate age at its own cutoff date.
PROFILE_CATEGORICAL_FIELDS = ['frequency', 'gender', 'region']
PROFILE_NUMERIC_FIELDS = [
    'num_inhabitants', 'ratio_urban', 'average_salary', 'unemployment_rate95',
    'unemployment_rate96', 'num_entrep_per1000', 'num_crimes95', 'num_crimes96',
]
display(profile[['account_id', 'create_date', 'frequency', 'birth_date', 'gender', 'region']].head())
display(profile[PROFILE_CATEGORICAL_FIELDS].nunique(dropna=False).to_frame('unique_values'))
print(f'Accounts with an owner profile: {profile.client_id.notna().mean():.1%}')

## Eligibility and account-disjoint split

A threshold of 20 events removes nearly empty ledgers but retains meaningful shorter histories. All transactions for an eligible account go to exactly one split. This is appropriate for MLM pre-training; a future loan/default task will also create a time cutoff inside each account history.

In [ ]:
MIN_EVENTS = 20
MIN_HISTORY_SPAN_DAYS = 14
TRAIN_FRACTION, VALID_FRACTION, TEST_FRACTION = 0.70, 0.15, 0.15
assert np.isclose(TRAIN_FRACTION + VALID_FRACTION + TEST_FRACTION, 1.0)

eligible_history = history.loc[
    (history['n_events'] >= MIN_EVENTS)
    & (history['history_span_days'] >= MIN_HISTORY_SPAN_DAYS)
].copy()

retention = pd.DataFrame({
    'accounts': [len(history), len(eligible_history)],
    'events': [history.n_events.sum(), eligible_history.n_events.sum()],
}, index=['all valid accounts', 'eligible accounts'])
retention['account_retained_%'] = 100 * retention['accounts'] / retention.iloc[0]['accounts']
retention['event_retained_%'] = 100 * retention['events'] / retention.iloc[0]['events']
display(retention)

account_ids = eligible_history['account_id'].to_numpy().copy()
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(account_ids)
train_end = round(len(account_ids) * TRAIN_FRACTION)
valid_end = train_end + round(len(account_ids) * VALID_FRACTION)
split_by_account = {
    **{account_id: 'train' for account_id in account_ids[:train_end]},
    **{account_id: 'valid' for account_id in account_ids[train_end:valid_end]},
    **{account_id: 'test' for account_id in account_ids[valid_end:]},
}

manifest = eligible_history.copy()
manifest['split'] = manifest['account_id'].map(split_by_account)
assert manifest['split'].notna().all()
display(manifest.groupby('split').agg(
    accounts=('account_id', 'nunique'),
    events=('n_events', 'sum'),
    median_events=('n_events', 'median'),
    median_span_days=('history_span_days', 'median'),
).reindex(['train', 'valid', 'test']))

## Save training artifacts

The exported Parquet files contain only eligible events/profile rows. Tokenizer fitting must read only `events_train.parquet` and `profile_train.parquet`.

In [ ]:
events_with_split = transactions.merge(
    manifest[['account_id', 'split']], on='account_id', how='inner', validate='many_to_one'
)
profile_with_split = profile.merge(
    manifest[['account_id', 'split']], on='account_id', how='inner', validate='one_to_one'
)

for split in ['train', 'valid', 'test']:
    events = events_with_split.loc[events_with_split['split'].eq(split)].sort_values(
        ['account_id', 'trans_date', 'trans_id']
    )
    profiles = profile_with_split.loc[profile_with_split['split'].eq(split)].sort_values('account_id')
    events.to_parquet(OUTPUT_DIR / f'events_{split}.parquet', index=False)
    profiles.to_parquet(OUTPUT_DIR / f'profile_{split}.parquet', index=False)

manifest.to_parquet(OUTPUT_DIR / 'account_split_manifest.parquet', index=False)
print(f'Saved events, profiles, and manifest to: {OUTPUT_DIR}')

# Guard against accidental account leakage.
account_sets = {
    split: set(events_with_split.loc[events_with_split['split'].eq(split), 'account_id'])
    for split in ['train', 'valid', 'test']
}
assert not (account_sets['train'] & account_sets['valid'])
assert not (account_sets['train'] & account_sets['test'])
assert not (account_sets['valid'] & account_sets['test'])
print('Account partitions are disjoint.')

## Tokenizer contract for the next notebook

Each raw event will become key/value pairs:

- numeric: `amount`, `balance` → field-specific percentile-bucket value tokens;
- categorical: `trans_type`, `operation`, `category`, `other_bank_id` → field-namespaced category tokens;
- time: `trans_date` → calendar features plus relative-time coordinates;
- never tokenize: `account_id`, `trans_id`, `other_account_id`, or `split`.

There is no free-text column in the base transaction table, so defer BPE. This is still a strong PRAGMA-like structured-event tokenizer and now its values have documented banking semantics.